In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ATMOS
  MANAGED LOCATION 'abfss://atmos-landing-dev-001@saatmosdevwestus20013.dfs.core.windows.net/';

CREATE SCHEMA   IF NOT EXISTS ATMOS.SILVER;

In [0]:
dbutils.widgets.text("ingestion_date", "", "Data de processamento (YYYY-MM-DD)")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, DateType, ArrayType, StructType, StructField, StringType

# Schema explícito do array "days" — necessário porque o Auto Loader
# armazena colunas JSON aninhadas como STRING na bronze.
days_schema = ArrayType(StructType([
    StructField("datetime",      StringType()),
    StructField("tempmax",       DoubleType()),
    StructField("tempmin",       DoubleType()),
    StructField("temp",          DoubleType()),
    StructField("dew",           DoubleType()),
    StructField("humidity",      DoubleType()),
    StructField("precip",        DoubleType()),
    StructField("windgust",      DoubleType()),
    StructField("windspeed",     DoubleType()),
    StructField("winddir",       DoubleType()),
    StructField("pressure",      DoubleType()),
    StructField("solarradiation",DoubleType()),
    StructField("uvindex",       DoubleType()),
    StructField("stations",      ArrayType(StringType())),
]))

In [0]:
df_bronze = (
    spark.table("atmos.bronze.visual_crossing_raw")
    .filter(F.col("ingestion_date") == ingestion_date)
)

In [0]:
df_exploded = (
    df_bronze
    .withColumn("days", F.from_json(F.col("days"), days_schema))
    .select(
        F.explode(F.col("days")).alias("day"),
        F.col("file_name"),
        F.col("source_system"),
        F.col("ingestion_timestamp"),
    )
)

In [0]:
df_normalized = (
    df_exploded
    .select(
        F.to_date(F.col("day.datetime")).cast(DateType()).alias("data_observacao"),
        F.lit(None).cast("string")                      .alias("hora_observacao"),
        F.col("day.stations").getItem(0)                .alias("id_estacao"),
        F.initcap(F.regexp_extract(F.col("file_name"), r"atmos_([^_/\]+)_\d+", 1)).alias("cidade"),
        F.col("source_system")                          .alias("sistema_origem"),
        F.col("day.temp")          .cast(DoubleType()).alias("temperatura_c"),
        F.col("day.tempmax")       .cast(DoubleType()).alias("temperatura_max_c"),
        F.col("day.tempmin")       .cast(DoubleType()).alias("temperatura_min_c"),
        F.col("day.dew")           .cast(DoubleType()).alias("ponto_orvalho_c"),
        F.col("day.humidity")      .cast(DoubleType()).alias("umidade_pct"),
        F.lit(None)                .cast(DoubleType()).alias("umidade_max_pct"),
        F.lit(None)                .cast(DoubleType()).alias("umidade_min_pct"),
        F.col("day.pressure")      .cast(DoubleType()).alias("pressao_hpa"),
        F.col("day.precip")        .cast(DoubleType()).alias("precipitacao_mm"),
        F.col("day.solarradiation").cast(DoubleType()).alias("radiacao_solar_wm2"),
        F.col("day.windspeed")     .cast(DoubleType()).alias("velocidade_vento_ms"),
        F.col("day.windgust")      .cast(DoubleType()).alias("rajada_vento_ms"),
        F.col("day.winddir")       .cast(DoubleType()).alias("direcao_vento_graus"),
        F.col("day.uvindex")       .cast(DoubleType()).alias("indice_uv"),
        F.to_date(F.lit(ingestion_date))                .alias("data_ingestao"),
    )
)

In [0]:
df_clean = (
    df_normalized
    .filter(F.col("data_observacao").isNotNull())
    .filter(F.col("id_estacao").isNotNull())
    .withColumn("temperatura_c",
        F.when(F.col("temperatura_c").between(-20, 50), F.col("temperatura_c")))
    .withColumn("temperatura_max_c",
        F.when(F.col("temperatura_max_c").between(-20, 55), F.col("temperatura_max_c")))
    .withColumn("umidade_pct",
        F.when(F.col("umidade_pct").between(0, 100), F.col("umidade_pct")))
    .withColumn("pressao_hpa",
        F.when(F.col("pressao_hpa").between(870, 1050), F.col("pressao_hpa")))
    .withColumn("precipitacao_mm",
        F.when(F.col("precipitacao_mm") < 0, F.lit(0.0)).otherwise(F.col("precipitacao_mm")))
    .withColumn("velocidade_vento_ms",
        F.when(F.col("velocidade_vento_ms").between(0, 100), F.col("velocidade_vento_ms")))
    .withColumn("radiacao_solar_wm2",
        F.when(F.col("radiacao_solar_wm2").between(0, 4000), F.col("radiacao_solar_wm2")))
    .dropDuplicates(["data_observacao", "id_estacao"])
)

In [0]:
(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"data_ingestao = '{ingestion_date}'")
    .partitionBy("data_ingestao")
    .saveAsTable("atmos.silver.climate_visual_crossing")
)

In [0]:
count = (
    spark.table("atmos.silver.climate_visual_crossing")
    .filter(F.col("data_ingestao") == ingestion_date)
    .count()
)

assert count > 0, f"Nenhum registro gravado em silver.climate_visual_crossing para data_ingestao={ingestion_date}."
print(f"OK — {count} registros em silver.climate_visual_crossing | data_ingestao={ingestion_date}")